In [1]:
import digitalhub as dh

project = dh.get_or_create_project("test-early-exit")

In [2]:
func = project.new_function(name="serve_function",
                            kind="python",
                            python_version="PYTHON3_10",
                            code_src="git+https://github.com/dlsab-ce/early-exit.git",
                            handler="inference_handler:handler",
                            init_function="init_model",
                            maxWorkers=1,
                            attributes={
                              "port":8080  
                            },
                            requirements=["torch>=2.2,<2.5", "torchaudio>=2.2,<2.5", "sentencepiece>=0.1.99",
                                "editdistance>=0.8.1", "tensorboard>=2.14", "huggingface_hub==1.11.0",
                                "flashlight==0.1.1", "flashlight-text==0.0.7", "soundfile==0.13.1"]
                           )

In [3]:
func = project.get_function("serve_function")

In [ ]:
build = func.run(
    action="build", 
    wait=True
)

In [18]:
run = func.run(action="serve", resources={"mem": "2Gi", "disk": "15Gi"}, init_parameters={"lang":"en"})

In [ ]:
stream_func = project.new_function("audiostream", kind="container", image="alexxit/go2rtc", command="/bin/bash", code_src="go2rtc")

In [ ]:
stream_func.run(action="serve", 
                service_ports=[{"port": 1984, "target_port": 1984}, {"port": 8554, "target_port": 8554}, {"port": 8555, "target_port": 8555}],
                args=["/shared/lunch_go2rtc.sh"],
                fs_group=8877, run_as_user=8877, run_as_group=887,
               resources={"mem": "1Gi", "disk": "1Gi"})

In [11]:
graph_func = project.new_function("early-exit-pipeline", kind="servicegraph", code_src="pipeline.yaml")

In [ ]:
graph_run = graph_func.run(action="serve", parameters={
    "input.url": "rtsp://s-test-early-exit-audiostream-latest.dev-platform:8554/webradio",
    "early-exit-service.url": "http://s-test-early-exit-servefunction-latest.dev-platform:8080"
}, service_ports=[{"port": 7777, "target_port": 7777}])